# file: 0_data_cleaning
# Purpose: Clean the crime statistics dataset

In [10]:
# Load Libraries
import pandas as pd
import numpy as np
#from functions.helper_funcs import encoder, unique_cols, corr_func
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
import matplotlib.pyplot as plt


In [11]:
# Variable Plot Function
def plot_variable(df, variable, title = ''):
    plt.hist(df[variable])
    plt.title(title)
    plt.show()

# Summary Stats Function
def summary_stats_table(dataframe):
    summary_stats = pd.DataFrame(index=dataframe.columns)
    summary_stats['count'] = dataframe.count()
    summary_stats['missing_values'] = dataframe.isnull().sum()
    summary_stats['unique_values'] = dataframe.nunique()
    summary_stats['data_type'] = dataframe.dtypes
    summary_stats['99th_percentile'] = dataframe.quantile(0.99)
    summary_stats['max'] = dataframe.max()
    summary_stats['min'] = dataframe.min()
    summary_stats['mean'] = dataframe.mean()
    summary_stats['std'] = dataframe.std()
    summary_stats['median'] = dataframe.median()
    summary_stats['skewness'] = dataframe.skew()
    summary_stats['kurtosis'] = dataframe.kurtosis()

    return summary_stats

# Encode Categorical Variables
from sklearn.preprocessing import LabelEncoder, OneHotEncoder
import pandas as pd

from sklearn.preprocessing import LabelEncoder, OneHotEncoder
import pandas as pd


def encoder(dataframe, encoding_method):
    encoded_dataframe = dataframe.copy()
    categorical_columns = dataframe.select_dtypes(include=['object', 'category']).columns
    # Store the non-categorical columns
    non_categorical_columns = dataframe.columns.difference(categorical_columns)
    if encoding_method == 'label':
        label_encoder = LabelEncoder()
        for column in categorical_columns:
            encoded_dataframe[column] = label_encoder.fit_transform(dataframe[column])
    elif encoding_method == 'one-hot':
        onehot_encoder = OneHotEncoder(sparse=False)
        encoded_array = onehot_encoder.fit_transform(dataframe[categorical_columns])
        feature_names = onehot_encoder.get_feature_names_out(categorical_columns)
        encoded_dataframe = pd.DataFrame(encoded_array, columns=feature_names)
    elif encoding_method == 'dummy':
        encoded_dataframe = pd.get_dummies(dataframe, columns=categorical_columns, drop_first=True)
    else:
        print("Invalid encoding method specified.")
        return None

    # Concatenate the encoded DataFrame with the non-categorical columns
    encoded_dataframe = pd.concat([encoded_dataframe, dataframe[non_categorical_columns]], axis=1)

    return encoded_dataframe

# Unique Cols
def unique_cols(df):
    less_than_10_unique = df.columns[df.nunique() < 10]
    subset_df = df[less_than_10_unique]
    return subset_df

# Correlation plot
def corr_func(dataframe, threshold):
    correlation_matrix = dataframe.corr()
    pairs = []
    for i in range(len(correlation_matrix.columns)):
        for j in range(i+1, len(correlation_matrix.columns)):
            variable1 = correlation_matrix.columns[i]
            variable2 = correlation_matrix.columns[j]
            correlation_coefficient = correlation_matrix.iloc[i, j]
            if abs(correlation_coefficient) > threshold:
                pairs.append((variable1, variable2, correlation_coefficient))
    if pairs:
        print("Pairs with correlation coefficient above threshold:")
        for pair in pairs:
            variable1, variable2, correlation_coefficient = pair
            print(f"{variable1} - {variable2}: {correlation_coefficient:.2f}")
    else:
        print("No pairs found with correlation coefficient above threshold.")



In [12]:
# Load Data
df = pd.read_csv('data/cime_data_2020.csv')
#print(df.head()[:2])

# Impute Missing Values
df = df.fillna(method = 'bfill')
df = df.fillna(method = 'ffill')
print(df.isna().sum())

# Rename
rename = {
    'DR_NO': 'div_record',
    'Date Rptd': 'report_date',
    'DATE OCC': 'incident_date',
    'TIME OCC': 'incident_time',
    'AREA': 'area',
    'AREA NAME': 'area_name',
    'Rpt Dist No': 'rpt',
    'Part 1-2': 'part',
    'Crm Cd': 'crime_code',
    'Crm Cd Desc': 'crime_code_def',
    'Vict Age': 'vict_age',
    'Vict Sex': 'vict_sex',
    'Vict Descent': 'vict_race',
    'Premis Desc': 'premise',
    'Weapon Used Cd': 'weapon_code',
    'Weapon Desc': 'weapon_def',
    'Status': 'case_status',
    'Status Desc': 'status_def',
    'Crm Cd 1': 'crime_1',
    'Crm Cd 2': 'crime_2',
    'Crm Cd 3': 'crime_3',
    'Crm Cd 4': 'crime_4',
    'LOCATION': 'location',
    'Cross Street': 'cross_street',
    'LAT': 'lat',
    'LON': 'lon'
}

# Rename Df
df = df.rename(columns=rename)
vars = [x for x in df.columns if 'date' in x or 'time' in x]
print(vars)

# Rename Columns and Convert to Date Time
date_time_vars = [x for x in df.columns if 'date' in x or 'time' in x]
df_transformed = df.copy()

for var in date_time_vars:
    df_transformed[var] = pd.to_datetime(df[var])
    df_transformed[f'year_{var}'] = df_transformed[var].dt.year
    df_transformed[f'month_{var}'] = df_transformed[var].dt.month
    df_transformed[f'day_{var}'] = df_transformed[var].dt.day
    df_transformed[f'hour_{var}'] = df_transformed[var].dt.hour
    df_transformed[f'minute_{var}'] = df_transformed[var].dt.minute

# Drop original columns and assign transformed DataFrame back to df
df = df_transformed.drop(date_time_vars, axis=1)

DR_NO             0
Date Rptd         0
DATE OCC          0
TIME OCC          0
AREA              0
AREA NAME         0
Rpt Dist No       0
Part 1-2          0
Crm Cd            0
Crm Cd Desc       0
Mocodes           0
Vict Age          0
Vict Sex          0
Vict Descent      0
Premis Cd         0
Premis Desc       0
Weapon Used Cd    0
Weapon Desc       0
Status            0
Status Desc       0
Crm Cd 1          0
Crm Cd 2          0
Crm Cd 3          0
Crm Cd 4          0
LOCATION          0
Cross Street      0
LAT               0
LON               0
dtype: int64
['report_date', 'incident_date', 'incident_time']


In [14]:
# Look at the values and their counts
df['crime_code_def'].value_counts().sort_values(ascending = False)[:10]

VEHICLE - STOLEN                                           79081
BATTERY - SIMPLE ASSAULT                                   58559
THEFT OF IDENTITY                                          47937
BURGLARY FROM VEHICLE                                      45810
VANDALISM - FELONY ($400 & OVER, ALL CHURCH VANDALISMS)    45392
BURGLARY                                                   44921
ASSAULT WITH DEADLY WEAPON, AGGRAVATED ASSAULT             42387
THEFT PLAIN - PETTY ($950 & UNDER)                         37878
INTIMATE PARTNER - SIMPLE ASSAULT                          37306
THEFT FROM MOTOR VEHICLE - PETTY ($950 & UNDER)            28903
Name: crime_code_def, dtype: int64

In [15]:
# Let's focus on Stolen Vehicle
df = df[df['crime_code_def'] == 'ASSAULT WITH DEADLY WEAPON, AGGRAVATED ASSAULT']
df.head(3)

,div_record,area,area_name,rpt,part,crime_code,crime_code_def,Mocodes,vict_age,vict_sex,...,year_incident_date,month_incident_date,day_incident_date,hour_incident_date,minute_incident_date,year_incident_time,month_incident_time,day_incident_time,hour_incident_time,minute_incident_time
24,200100546,1,Central,166,1,230,"ASSAULT WITH DEADLY WEAPON, AGGRAVATED ASSAULT",0416 0913 2004 1218,62,M,...,2020,1,15,0,0,1970,1,1,0,0
27,200100552,1,Central,111,1,230,"ASSAULT WITH DEADLY WEAPON, AGGRAVATED ASSAULT",2004 0305 0411 1202 1414 1402 1822 0906,71,M,...,2020,1,19,0,0,1970,1,1,0,0
42,200100578,1,Central,163,1,230,"ASSAULT WITH DEADLY WEAPON, AGGRAVATED ASSAULT",0947 0913 0416 0447 1402 0319 0443 1420 2004,52,M,...,2020,1,30,0,0,1970,1,1,0,0


In [16]:
# Save the Data Out
df.to_csv('data/eda_data_deadly.csv', index = False)

# HELP HERE ###


In [8]:
# Drop Highly Correlated Variables
#corr_func(df, threshold=.75)
drop_cols = ['rpt', 'crime_code', 'lat', 'year_report_date', 'month_report_date']
df = df.drop(drop_cols, axis=1, errors='ignore')

# Large Columns
cat_cols = [col for col in df.columns if df[col].dtype == 'object' and col != 'vict_sex']
large_cols = [x for x in cat_cols if df[x].nunique() > 100]

# Drop Large Cols
df = df.drop(large_cols, axis=1)
df = df.drop(['div_record', 'status_def'], axis=1)
cat_cols = [col for col in df.columns if df[col].dtype == 'object' and col != 'vict_sex']

# Take log Numeric Cols
contin = [x for x in df.columns if x not in cat_cols and x != 'vict_sex']
print(contin)

# Scale Data
scaler = StandardScaler()
df[contin] = scaler.fit_transform(df[contin])

# Encode Categorical Variables
df_encoded = encoder(df[cat_cols], encoding_method='one-hot')  # Encode categorical columns
df = pd.concat([df.drop(cat_cols, axis=1), df_encoded], axis=1)

# Correlation function on DF
corr_func(df, threshold=0.75)

# PCA Analysis
df_target = df[['vict_age']]
df = df.drop('vict_age', axis=1)

# Instantiate PCA
pca = PCA(n_components=10)
pca.fit(df)
pca_df = pca.transform(df)

# Append pca_df to df_target
df_pca = pd.DataFrame(pca_df, columns=[f'PC{i+1}' for i in range(pca_df.shape[1])])
df = pd.concat([df_pca, df_target], axis=1)

# Save Output
df.to_csv('data/crime_stats_cleaned.csv', index=False)

DR_NO             0
Date Rptd         0
DATE OCC          0
TIME OCC          0
AREA              0
AREA NAME         0
Rpt Dist No       0
Part 1-2          0
Crm Cd            0
Crm Cd Desc       0
Mocodes           0
Vict Age          0
Vict Sex          0
Vict Descent      0
Premis Cd         0
Premis Desc       0
Weapon Used Cd    0
Weapon Desc       0
Status            0
Status Desc       0
Crm Cd 1          0
Crm Cd 2          0
Crm Cd 3          0
Crm Cd 4          0
LOCATION          0
Cross Street      0
LAT               0
LON               0
dtype: int64
['report_date', 'incident_date', 'incident_time']
['area', 'part', 'vict_age', 'Premis Cd', 'weapon_code', 'crime_1', 'crime_2', 'crime_3', 'crime_4', 'lon', 'day_report_date', 'hour_report_date', 'minute_report_date', 'year_incident_date', 'month_incident_date', 'day_incident_date', 'hour_incident_date', 'minute_incident_date', 'year_incident_time', 'month_incident_time', 'day_incident_time', 'hour_incident_time', 'minute_

AttributeError: 'OneHotEncoder' object has no attribute 'get_feature_names_out'